# LayoutVLM 完整复现 - CVPR 2025

**论文**: [LayoutVLM: Differentiable Optimization of 3D Layout via Vision-Language Models](https://arxiv.org/abs/2412.02193)

**GitHub**: https://github.com/sunfanyunn/LayoutVLM

---

## 🎯 功能说明

此notebook整合了完整的工作流程：
1. ✅ 在Colab中安装Blender 4.2.1
2. ✅ 配置GPU渲染
3. ✅ 运行LayoutVLM生成3D布局
4. ✅ 自动渲染可视化结果

---

## 📋 使用步骤

1. **启用GPU**: 运行时 → 更改运行时类型 → T4 GPU
2. **高RAM（推荐）**: 运行时 → 更改运行时类型 → 高RAM
3. **按顺序执行**所有单元格
4. **配置API**: 在步骤8中填入你的API密钥

⏱️ **预计时间**: 首次运行约15-20分钟（后续约5-10分钟）

---

## 步骤 1️⃣: 检查GPU环境

In [ ]:
# 检查GPU
print('='*60)
print('🔍 检查GPU环境')
print('='*60)

!nvidia-smi

print('\n' + '='*60)
print('✅ GPU检查完成')
print('='*60)
print('\n⚠️  如果看不到GPU信息，请检查运行时设置')

## 步骤 2️⃣: 挂载Google Drive

💾 用于永久保存数据集和结果

In [ ]:
from google.colab import drive
import os

# 挂载Drive
drive.mount('/content/drive')

# 创建项目目录结构
PROJECT_DIR = '/content/drive/MyDrive/LayoutVLM_Project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/datasets', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/blender_scripts', exist_ok=True)

print('\n' + '='*60)
print('✅ Google Drive已挂载')
print('='*60)
print(f'📁 项目目录: {PROJECT_DIR}')
print(f'📊 结果目录: {PROJECT_DIR}/results')
print(f'💾 数据集目录: {PROJECT_DIR}/datasets')
print('='*60)

## 步骤 3️⃣: 安装Blender 4.2.1

🔧 使用官方二进制包，支持完整的Python API

In [ ]:
import os

print('='*60)
print('🔧 安装Blender 4.2.1')
print('='*60)

# 1. 安装系统依赖
print('\n📦 步骤1/4: 安装系统依赖...')
!apt-get update -y -qq > /dev/null 2>&1
!apt-get install -y -qq xvfb libegl1-mesa libxrandr2 libxinerama1 libxxf86vm1 libxi6 wget > /dev/null 2>&1
print('   ✅ 系统依赖安装完成')

# 2. 下载Blender
print('\n📥 步骤2/4: 下载Blender 4.2.1...')
BLENDER_URL = "https://download.blender.org/release/Blender4.2/blender-4.2.1-linux-x64.tar.xz"
BLENDER_FILE = "blender-4.2.1-linux-x64.tar.xz"

if not os.path.exists(f'/content/{BLENDER_FILE}'):
    !wget -q --show-progress {BLENDER_URL}
    print('   ✅ 下载完成')
else:
    print('   ✅ Blender压缩包已存在')

# 3. 解压Blender
print('\n📂 步骤3/4: 解压Blender...')
if not os.path.exists('/content/blender-4.2.1-linux-x64'):
    !tar -xJf {BLENDER_FILE}
    print('   ✅ 解压完成')
else:
    print('   ✅ Blender已解压')

# 4. 设置环境变量
BLENDER = "/content/blender-4.2.1-linux-x64/blender"
os.environ['BLENDER_PATH'] = BLENDER

# 5. 验证安装
print('\n🔍 步骤4/4: 验证Blender安装...')
!$BLENDER --version

print('\n' + '='*60)
print('✅ Blender 4.2.1 安装完成')
print(f'📍 位置: {BLENDER}')
print('='*60)

## 步骤 4️⃣: 创建GPU配置脚本

⚡ 启用GPU加速渲染

In [ ]:
# 创建GPU配置脚本（基于你的set_cycles_gpu.py）
gpu_script = '''import bpy

# 配置Cycles渲染引擎使用GPU
prefs = bpy.context.preferences.addons["cycles"].preferences

# 尝试OPTIX，如果不支持则使用CUDA
try:
    prefs.compute_device_type = "OPTIX"
    print("✅ 使用OPTIX")
except Exception:
    prefs.compute_device_type = "CUDA"
    print("✅ 使用CUDA")

# 获取所有可用设备
prefs.get_devices()

# 启用所有GPU
gpu_count = 0
for dev in prefs.devices:
    try:
        dev.use = True
        if dev.type in ["CUDA", "OPTIX"]:
            gpu_count += 1
            print(f"   GPU {gpu_count}: {dev.name}")
    except:
        pass

# 设置场景使用GPU
bpy.context.scene.cycles.device = "GPU"

print(f"\\n[Cycles] 计算设备类型: {prefs.compute_device_type}")
print(f"[Cycles] 启用GPU数量: {gpu_count}")
print("✅ GPU渲染配置完成")
'''

# 保存到本地和Drive
with open('/content/set_cycles_gpu.py', 'w') as f:
    f.write(gpu_script)

import shutil
shutil.copy('/content/set_cycles_gpu.py', f'{PROJECT_DIR}/blender_scripts/set_cycles_gpu.py')

print('='*60)
print('✅ GPU配置脚本已创建')
print('='*60)
print('📄 本地: /content/set_cycles_gpu.py')
print(f'💾 备份: {PROJECT_DIR}/blender_scripts/set_cycles_gpu.py')
print('='*60)

## 步骤 5️⃣: 克隆LayoutVLM

📥 从GitHub获取最新代码

In [ ]:
import os

os.chdir('/content')

print('='*60)
print('📥 克隆LayoutVLM仓库')
print('='*60)

# 清理旧版本
if os.path.exists('LayoutVLM'):
    print('\n🗑️  清理旧版本...')
    !rm -rf LayoutVLM

# 克隆仓库
print('\n📦 正在克隆...')
!git clone -q https://github.com/sunfanyunn/LayoutVLM.git

os.chdir('/content/LayoutVLM')

print('\n' + '='*60)
print('✅ LayoutVLM已克隆')
print('='*60)
print(f'📍 位置: {os.getcwd()}')
print('\n📂 项目结构:')
!ls -1

## 步骤 6️⃣: 安装Python依赖到Blender

📦 **关键步骤**: 将依赖包安装到Blender的Python环境

In [ ]:
import os

BLENDER = os.environ['BLENDER_PATH']

print('='*60)
print('📦 安装Python依赖到Blender')
print('='*60)
print('\n⏱️  这可能需要3-5分钟，请耐心等待...\n')

# 创建依赖安装脚本
install_script = '''import sys
import subprocess

# 需要安装的包
packages = [
    "openai",
    "langchain",
    "langchain-openai",
    "langchain-core",
    "langchain-community",
    "numpy",
    "torch",
    "torchvision",
    "trimesh",
    "Pillow",
    "tiktoken",
    "pyyaml",
    "tqdm"
]

print(f"Python路径: {sys.executable}")
print(f"Python版本: {sys.version.split()[0]}")
print(f"\\n开始安装 {len(packages)} 个依赖包...\\n")

success = 0
failed = []

for i, package in enumerate(packages, 1):
    print(f"[{i}/{len(packages)}] 安装 {package}...", end=" ")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location", package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        print("✅")
        success += 1
    except Exception as e:
        print(f"❌ ({str(e)[:30]}...)")
        failed.append(package)

print("\\n" + "="*60)
print(f"✅ 成功: {success}/{len(packages)}")
if failed:
    print(f"❌ 失败: {len(failed)} - {', '.join(failed)}")
print("="*60)
'''

# 保存安装脚本
with open('/tmp/install_deps_blender.py', 'w') as f:
    f.write(install_script)

# 使用Blender的Python执行安装
!$BLENDER --background --python /tmp/install_deps_blender.py

print('\n' + '='*60)
print('✅ 依赖安装完成')
print('='*60)

## 步骤 7️⃣: 编译CUDA扩展

⚙️ 编译Rotated IOU Loss（用于边界框优化）

In [ ]:
import os
import sys
import torch

print('='*60)
print('⚙️  编译CUDA扩展 (Rotated IoU Loss)')
print('='*60)

# ========== 第1步: 检查环境 ==========
print('\n📊 第1步: 检查编译环境')
print(f'   系统 Python: {sys.executable}')
print(f'   Python 版本: {sys.version.split()[0]}')
print(f'   PyTorch: {torch.__version__}')
print(f'   CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   CUDA 版本: {torch.version.cuda}')
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

# ========== 第2步: 安装编译依赖 ==========
print('\n📦 第2步: 安装编译依赖')
!pip install -q ninja

# ========== 第3步: 编译 CUDA 扩展 ==========
print('\n🔨 第3步: 编译 CUDA 扩展 (约1-2分钟)')
print('   ⚠️  关键: 使用系统 Python3 而非 Blender Python')
print('   原因: Blender Python 缺少 Python.h 开发头文件\n')

os.chdir('/content/LayoutVLM/third_party/Rotated_IoU/cuda_op')

# 使用系统 Python3 编译（确保有完整开发环境）
!python3 setup.py install --user 2>&1

os.chdir('/content/LayoutVLM')

# ========== 第4步: 添加模块路径 ==========
print('\n📂 第4步: 配置模块路径')

# 添加用户安装路径到 sys.path
import site
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
    print(f'   已添加: {user_site}')

# 也添加 root 用户的本地路径
root_local = '/root/.local/lib/python3.12/site-packages/sort_vertices-0.0.0-py3.12-linux-x86_64.egg'
if root_local not in sys.path:
    sys.path.insert(0, root_local)
    print(f'   已添加: {root_local}')

# ========== 第5步: 验证编译结果 ==========
print('\n' + '='*60)
print('🔍 第5步: 验证 CUDA 扩展')
print('='*60)

try:
    # 重新加载模块（如果之前导入过）
    import importlib
    if 'sort_vertices' in sys.modules:
        importlib.reload(sys.modules['sort_vertices'])
    
    # 测试导入 sort_vertices 模块
    import sort_vertices
    print('✅ sort_vertices 模块导入成功')
    print(f'   模块路径: {sort_vertices.__file__}')
    
    # 测试 oriented_iou_loss 功能
    sys.path.insert(0, '/content/LayoutVLM')
    from third_party.Rotated_IoU import oriented_iou_loss
    print('✅ oriented_iou_loss 导入成功')
    
    # 功能测试
    test_boxes1 = torch.randn(2, 5).cuda()
    test_boxes2 = torch.randn(2, 5).cuda()
    loss = oriented_iou_loss(test_boxes1, test_boxes2)
    print(f'✅ CUDA IoU 计算成功: loss = {loss.item():.4f}')
    
    # 检查全局标志
    from src.layoutvlm import constraints
    print(f'✅ ORIENTED_IOU_AVAILABLE = {constraints.ORIENTED_IOU_AVAILABLE}')
    
    print('\n' + '='*60)
    print('🎉 CUDA 扩展配置完美！')
    print('='*60)
    print('💡 性能提升: CUDA 比 CPU 快 5-10 倍')
    
except ImportError as e:
    print(f'\n❌ 导入失败: {e}')
    print(f'\n🔍 当前 sys.path:')
    for p in sys.path[:5]:
        print(f'   - {p}')
    print('\n⚠️  CUDA 扩展未成功编译')
    print('💡 系统将使用 CPU 后备方案 (Shapely)')
    print('   性能影响: 优化速度会慢 5-10 倍')
    print('\n🔧 故障排查:')
    print('   1. 检查上面的编译输出是否有错误')
    print('   2. 运行: !find /root/.local -name "sort_vertices*"')
    print('   3. 运行诊断脚本: !python3 /content/LayoutVLM/check_cuda_extension.py')
except Exception as e:
    print(f'\n❌ 测试失败: {e}')
    import traceback
    print('详细错误:')
    traceback.print_exc()

print('\n' + '='*60)
print('✅ 步骤7完成')
print('='*60)

## 步骤 8️⃣: 准备数据集

💾 下载Objaverse资产数据集（约2.4GB）

In [ ]:
import os

DATASET_PATH = f'{PROJECT_DIR}/datasets/dataset.zip'

print('='*60)
print('💾 准备数据集')
print('='*60)

# 检查Drive中是否已有数据集
if os.path.exists(DATASET_PATH):
    print('\n✅ 数据集已存在于Drive，直接复制...')
    !cp {DATASET_PATH} /content/LayoutVLM/dataset.zip
    print('   ✅ 复制完成')
else:
    print('\n📥 首次运行，正在下载数据集（约2.4GB）...')
    print('   这可能需要3-5分钟...\n')
    
    !pip install -q gdown
    !gdown 1WGbj8gWn-f-BRwqPKfoY06budBzgM0pu -O /content/LayoutVLM/dataset.zip
    
    print('\n💾 备份数据集到Drive（下次运行更快）...')
    !cp /content/LayoutVLM/dataset.zip {DATASET_PATH}
    print('   ✅ 备份完成')

# 解压数据集
print('\n📂 解压数据集...')
!mkdir -p /content/LayoutVLM/data
!unzip -q /content/LayoutVLM/dataset.zip -d /content/LayoutVLM/data/

# 检查解压结果
print('\n📊 数据集内容:')
!ls -lh /content/LayoutVLM/data/ | head -10

print('\n' + '='*60)
print('✅ 数据集准备完成')
print('='*60)

## 步骤 9️⃣: 配置转接API

🔑 **重要**: 填入你的API密钥和配置

In [ ]:
import os

print('='*60)
print('🔑 配置转接API')
print('='*60)

# ⚠️⚠️⚠️ 在这里填入你的配置 ⚠️⚠️⚠️
API_KEY = "sk-YOUR_API_KEY_HERE"  # 替换成你的API密钥！
BASE_URL = "https://chat.cloudapi.vip/v1/"
MODEL_NAME = "gpt-4o"  # 或其他支持vision的模型

# 设置环境变量
os.environ['OPENAI_API_KEY'] = API_KEY
os.environ['OPENAI_BASE_URL'] = BASE_URL

print('\n📋 API配置信息:')
print(f'   🔑 API Key: {API_KEY[:15]}...')
print(f'   🌐 Base URL: {BASE_URL}')
print(f'   🤖 Model: {MODEL_NAME}')

# 测试API连接
print('\n🧪 测试API连接...')
try:
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": "测试"}],
        max_tokens=5
    )
    print('   ✅ API连接成功')
    print(f'   📨 响应: {response.choices[0].message.content}')
except Exception as e:
    print(f'   ❌ API测试失败: {e}')
    print('   ⚠️  请检查API Key和Base URL是否正确')

print('\n' + '='*60)
print('✅ API配置完成')
print('='*60)

## 步骤 🔟: 创建场景配置

📝 定义要生成的3D场景

In [ ]:
import json
import os

os.chdir('/content/LayoutVLM')

print('='*60)
print('📝 创建场景配置')
print('='*60)

# 创建示例场景
scene_config = {
    "task_description": "Create a cozy living room with modern furniture",
    "layout_criteria": "Natural furniture arrangement with good flow and spacing. Sofa against the wall, coffee table in front of it.",
    "boundary": {
        "floor_vertices": [
            [0, 0, 0],
            [6, 0, 0],
            [6, 0, 5],
            [0, 0, 5]
        ],
        "wall_height": 3.0
    },
    "assets": {
        # 这里会由LayoutVLM根据数据集自动填充
        # 或者你可以从benchmark_tasks中复制示例
    }
}

# 保存配置
with open('scene_config.json', 'w') as f:
    json.dump(scene_config, f, indent=2)

print('\n✅ 场景配置已创建')
print('\n📄 配置内容:')
print(json.dumps(scene_config, indent=2))

print('\n' + '='*60)
print('💡 提示: 你可以查看 benchmark_tasks/ 目录')
print('   获取更多场景配置示例')
print('='*60)

# 显示可用的示例
print('\n📚 可用的benchmark示例:')
!find benchmark_tasks -name "*.json" | head -5

## 步骤 1️⃣1️⃣: 运行LayoutVLM

🚀 **核心步骤**: 使用Blender Python运行LayoutVLM生成布局

⏱️ 预计时间: 5-15分钟（取决于场景复杂度）

In [ ]:
import os

BLENDER = os.environ['BLENDER_PATH']
os.chdir('/content/LayoutVLM')

print('='*60)
print('🚀 运行LayoutVLM')
print('='*60)
print('\n⏱️  这可能需要5-15分钟，请耐心等待...')
print('💡 你可以在下方看到实时进度\n')

# 创建运行脚本
run_script = f'''import os
import sys

# 设置环境变量
os.environ["OPENAI_API_KEY"] = "{API_KEY}"
os.environ["OPENAI_BASE_URL"] = "{BASE_URL}"

# 添加路径
sys.path.insert(0, "/content/LayoutVLM")

# 设置命令行参数
sys.argv = [
    "main.py",
    "--scene_json_file", "scene_config.json",
    "--openai_api_key", "{API_KEY}",
    "--save_dir", "{PROJECT_DIR}/results"
]

print("="*60)
print("🎨 LayoutVLM 开始生成布局")
print("="*60)
print()

# 执行main.py
try:
    with open("/content/LayoutVLM/main.py", "r") as f:
        exec(f.read())
    print()
    print("="*60)
    print("✅ LayoutVLM执行完成")
    print("="*60)
except Exception as e:
    print()
    print("="*60)
    print(f"❌ 执行出错: {{e}}")
    print("="*60)
    import traceback
    traceback.print_exc()
'''

# 保存运行脚本
with open('/tmp/run_layoutvlm.py', 'w') as f:
    f.write(run_script)

print('='*60)
print('开始执行...')
print('='*60)
print()

# 使用xvfb和Blender运行
!xvfb-run -a $BLENDER --background --python /tmp/run_layoutvlm.py

print()
print('='*60)
print('✅ 运行完成！')
print('='*60)
print(f'📁 结果保存在: {PROJECT_DIR}/results')
print('='*60)

## 步骤 1️⃣2️⃣: 查看生成结果

📊 查看生成的布局数据和渲染图片

In [ ]:
import os
import json
from IPython.display import Image, display
import glob

result_dir = f'{PROJECT_DIR}/results'

print('='*60)
print('📊 查看生成结果')
print('='*60)

# 列出所有生成的文件
print('\n📁 生成的文件:')
!ls -lh {result_dir}

# 读取布局JSON
layout_file = f'{result_dir}/layout.json'

if os.path.exists(layout_file):
    print('\n' + '='*60)
    print('📋 布局数据 (layout.json):')
    print('='*60)
    
    with open(layout_file, 'r') as f:
        layout = json.load(f)
    
    # 显示简要信息
    print(f'\n✅ 成功生成 {len(layout)} 个物体的布局')
    
    # 显示前几个物体的信息
    print('\n前3个物体的信息:')
    for i, (obj_id, obj_info) in enumerate(list(layout.items())[:3], 1):
        print(f'\n{i}. {obj_id}:')
        print(f'   位置: {obj_info.get("position", "N/A")}')
        print(f'   旋转: {obj_info.get("rotation", "N/A")}')
        if len(layout) > 3 and i == 3:
            print(f'\n... 还有 {len(layout) - 3} 个物体')
    
    # 完整JSON预览
    print('\n完整JSON数据（前500字符）:')
    json_str = json.dumps(layout, indent=2)
    print(json_str[:500])
    if len(json_str) > 500:
        print(f'\n... (还有 {len(json_str) - 500} 字符)')
    
else:
    print('\n❌ 未找到 layout.json 文件')
    print('   请检查运行日志中的错误信息')

# 查找并显示渲染图片
print('\n' + '='*60)
print('🖼️  渲染图片:')
print('='*60)

image_files = glob.glob(f'{result_dir}/*.png') + glob.glob(f'{result_dir}/*.jpg')

if image_files:
    print(f'\n找到 {len(image_files)} 张图片:\n')
    for img_path in image_files[:5]:  # 最多显示5张
        print(f'📷 {os.path.basename(img_path)}')
        try:
            display(Image(filename=img_path, width=600))
            print()
        except:
            print(f'   ⚠️  无法显示图片')
    
    if len(image_files) > 5:
        print(f'\n... 还有 {len(image_files) - 5} 张图片')
else:
    print('\n⚠️  未找到渲染图片')
    print('   可能渲染功能未启用或发生错误')

print('\n' + '='*60)
print('✅ 结果查看完成')
print('='*60)
print(f'\n💾 所有结果已保存在: {result_dir}')
print('   你可以在Google Drive中查看完整结果')

---

## 🔧 故障排查

### 常见问题

#### 1. API连接失败
- 检查API Key是否正确
- 确认Base URL格式正确（包含 `/v1/`）
- 检查账户余额

#### 2. 内存不足
- 尝试使用高RAM运行时
- 减小场景复杂度

#### 3. GPU相关错误
- 确认已启用GPU运行时
- 检查CUDA版本兼容性

#### 4. 依赖安装失败
- 重新运行步骤6
- 检查网络连接

### 重新运行

如果需要重新运行：
1. **完全重启**: 运行时 → 重启运行时
2. **从头执行**: 按顺序重新运行所有单元格
3. **数据集会从Drive快速恢复**（无需重新下载）

---

## 📚 参考资源

- **论文**: [LayoutVLM: Differentiable Optimization of 3D Layout via Vision-Language Models](https://arxiv.org/abs/2412.02193)
- **GitHub**: https://github.com/sunfanyunn/LayoutVLM
- **项目主页**: https://ai.stanford.edu/~sunfanyun/layoutvlm/
- **CVPR 2025**: Proceedings (June 2025)

---

## 💡 下一步

1. **尝试不同场景**: 修改`scene_config.json`
2. **使用benchmark**: 复制`benchmark_tasks`中的示例
3. **调整参数**: 修改布局标准和边界
4. **自定义资产**: 添加自己的3D模型

---